## Video 3: Prepropressing

__Plan__:
1. Apply preprocessing steps
2. Visualise what we've done
3. Save our processed data, using different formats and parallel processing
   

__Resources__:

[List of preprocessing steps in spikeinterface](https://spikeinterface.readthedocs.io/en/latest/api.html#module-spikeinterface.preprocessing) 

[Advanced tutorial, including information about parallel processing and how spikeinterface stores information](https://github.com/SpikeInterface/SpikeInterface-Training-Edinburgh-May24/tree/main/hands_on/cookbooks) and [Youtube video of tutorial](https://youtu.be/pHze_8s4Qak?si=Qkntrs0TVbnliJ6A) \
[Advanced tutorial about drift/motion correction](https://github.com/SpikeInterface/SpikeInterface-Training-Edinburgh-May24/tree/main/hands_on/handling_drift) and [YouTube video](https://youtu.be/pHze_8s4Qak?si=pwuqItRm9cYRaRvu&t=2854) \
[Advanced tutorial on preprocessing, focused on Neuropixels recordings](https://github.com/SpikeInterface/SpikeInterface-Training-Edinburgh-May24/tree/main/hands_on/preprocessing) and [YouTube video]() 

[Talk about the International Brain Lab's pipeline (including lots of preprocessing!)](https://youtu.be/Uvn0a5DL7K0?si=YGtgrWbkvKNxqM3Q) \
[IBL's white paper on their spike sorting pipeline](https://figshare.com/articles/online_resource/Spike_sorting_pipeline_for_the_International_Brain_Laboratory/19705522?file=35040628)


Now that we have a recording with a probe attached, let's do some preprocessing! First import spikeinterface, read in your recording (I'm using a three minute sample that I made earlier and saved as a binary folder) and check you've got a probe.

In [ ]:
import spikeinterface.full as si
path_to_recording = "/Users/chris/Work/Edinburgh/Spike/Training/L4_sort/three_minute_sample_bin"
recording = si.read_binary_folder(path_to_recording)
print(recording.has_probe())

We can visualise the raw data using `plot_traces`. We'll try and use the `ipywidgets` backend (for installation guidance: https://spikeinterface.readthedocs.io/en/stable/modules/widgets.html). This should generate a cool, interactive GUI in your Jupyter Notebook

In [ ]:
%matplotlib widget
si.plot_traces(recording, backend="ipywidgets")

The raw traces aren't super interesting. They're a bit nicer if we preprocess first. Let's apply a `highpass_filter`

In [ ]:
highpass_recording = si.highpass_filter(recording)

Another common preprocessing step is to do a common value reference. We're going to try out two `common_reference` options, one using the default argument (the median value) and one using the `average` operator instead. Then we can visualise the difference between all these preprocessed recording using `plot_traces`.

In [ ]:
cr_recording = si.common_reference(highpass_recording)
cr_avg_recording = si.common_reference(highpass_recording, operator="average")

In [ ]:
si.plot_traces(
    recording = {
        'highpass': highpass_recording,
        'median': cr_recording,
        'avg': cr_avg_recording
    },
    backend="ipywidgets",
    channel_ids=['CH50','CH51','CH52'], # You'll need to change this to be some of the channel_ids in your recording. Or remove this line.
    time_range=(6,7), # Only look at traces from tiem 6 to time 7
    order_channel_by_depth=True
)

There are many possible preprocessing steps: including motion correction, bad channel detection, whitening and more! See more here: https://spikeinterface.readthedocs.io/en/latest/api.html#module-spikeinterface.preprocessing or here: https://spikeinterface.readthedocs.io/en/stable/how_to/handle_drift.html

Depending on your workflow, you might want to save your preprocessed recording before doing any sorting. You can do this using the `save_to_folder`, which will save your recording as a binary folder (basically a bit `.bin` file with lots of metadata) or `save_to_zarr` which will compress the recording. 

In [ ]:
project_path = "/Users/chris/Work/Edinburgh/Spike/Training/L3_pre/outputs/"

In [ ]:
cr_recording.save_to_folder(folder=project_path + "preprocessed_recording")

In [ ]:
cr_recording.save_to_zarr(folder=project_path + "preprocessed_recording_compressed")

We can speed this, and many other bits, of spikeinterface up by increasing the number of jobs.

In [ ]:
si.set_global_job_kwargs(n_jobs=4)
cr_recording.save_to_zarr(folder=project_path + "preprocessed_recording_compressed_again")

There are quite a lot of job settings to play with, and what is best can depend on your system (the speed of your harddrive, which Operating System you use etc). Have a play to find out which is best for your set up!

In [ ]:
si.set_global_job_kwargs(pool_engine = "process", mp_context="spawn", n_jobs=2)
cr_recording.save_to_folder(folder=project_path + "preprocessed_recording_compressed_4")